## From Sound Wave to Music Genre: Recognition of Unstructured Audio

**Topic:** Preparation of data for genre classification directly from audio files.

**Goal:** To create clean and organized groups of data (for training, validation, and testing) from the fma_small package. This data will be necessary for the next notebook, where the artificial intelligence itself will be trained.

**What this notebook does:**

1. Downloads the data archive (342 MB) and verifies that it was downloaded correctly.
2. Extracts information about the songs (which song belongs to which of the 8 genres).
3. Selects exactly 100 songs from each genre to ensure our model remains objective.
4. Divides the songs into three groups: for training (70%), for validation (15%), and for a final test (15%).
5. Records these lists and a settings file (manifest), so that we can repeat the same experiment at any time.

**Dataset.** FMA (Defferrard et al., ISMIR 2017). Metadata license: CC BY 4.0;
audio: per-artist Creative Commons (research use).

### 1. Ensuring identical results

Fixed settings (seed), software versions, and file checksums are set. This ensures that every subsequent run of the notebook will produce absolutely the same results.

In [1]:
import os, random, hashlib, json
import numpy as np

# single global seed reused everywhere so all runs are identical
SEED = 42

def set_seed(seed: int = SEED) -> None:
    # make Python hash-based ops deterministic
    os.environ["PYTHONHASHSEED"] = str(seed)
    # seed the standard-library RNG (Random Number Generator)
    random.seed(seed)
    # seed NumPy's RNG
    np.random.seed(seed)

def package_versions() -> dict:
    # map package name -> version string for the reproducibility record
    versions = {}
    for name in ["numpy", "librosa", "sklearn", "torch", "matplotlib"]:
        try:
            module = __import__(name)
            versions[name] = getattr(module, "__version__", "unknown")
        except ImportError:
            versions[name] = "not installed"
    return versions

def file_checksum(path, algo: str = "sha256") -> str:
    # checksum to prove the data has not changed ('sha1' for FMA archives)
    h = hashlib.new(algo)
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def save_manifest(manifest: dict, path) -> None:
    # save checksums / versions as pretty JSON
    with open(path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)

### 2. Configuration

Paths, data source URL, the official checksum, and the number of tracks per genre retained (so the full 7.2 GB audio archive is not required).

In [ ]:
import urllib.request, zipfile
import pandas as pd
from sklearn.model_selection import train_test_split

# root directory where all data will be stored locally
DATA_DIR = "data"
# public URL of the FMA metadata archive (342 MB, contains tracks.csv)
METADATA_URL = "https://os.unil.cloud.switch.ch/fma/fma_metadata.zip"
# official SHA-1 checksum of fma_metadata.zip (from the FMA README)
METADATA_SHA1 = "f0df49ffe5f2a6008d7dc83c6915b31835dfe733"
# how many tracks per genre we keep (avoids using the full 7.2 GB audio)
TRACKS_PER_GENRE = 100

### 3. Download & verify metadata (342 MB — not the 7.2 GB audio)
Downloads the metadata archive once, checks its SHA-1 to guarantee integrity,
then extracts `tracks.csv`. Both steps are idempotent (safe to re-run).

In [ ]:
def download_metadata():
    print("[1/5] Metadata: checking / downloading fma_metadata.zip ...")
    zip_path = os.path.join(DATA_DIR, "fma_metadata.zip")
    os.makedirs(DATA_DIR, exist_ok=True)
    # Idempotent: download only if not already present
    if not os.path.exists(zip_path):
        print("      -> downloading ~342 MB (may take a few minutes)...")
        urllib.request.urlretrieve(METADATA_URL, zip_path)
    else:
        print("      -> already present, skipping download.")
    # Verify integrity against the official checksum
    print("      -> verifying SHA-1 checksum ...")
    assert file_checksum(zip_path, algo="sha1") == METADATA_SHA1, "Checksum mismatch!"
    print("      -> checksum OK.")
    return zip_path

def extract_metadata(zip_path):
    print("[2/5] Extracting metadata archive ...")
    tracks_csv = os.path.join(DATA_DIR, "fma_metadata", "tracks.csv")
    if os.path.exists(tracks_csv):
        print("      -> tracks.csv already extracted, skipping.")
        return tracks_csv
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(DATA_DIR)
    return tracks_csv

### 4. Labels, balanced selection, and proportional data groups
Load the mapping from track ID to genre, select a fixed subset of 100 tracks for each genre to keep the dataset balanced, and then split those tracks into training, validation, and test sets so that each split preserves the same class proportions.

In [ ]:
def load_labels(tracks_csv):
    print("[3/5] Loading labels from tracks.csv ...")
    # tracks.csv has a 2-level column header
    tracks = pd.read_csv(tracks_csv, index_col=0, header=[0, 1])
    # кeep only the 'small' subset (8 balanced genres)
    small = tracks[tracks[("set", "subset")] == "small"]
    labels = pd.DataFrame({
        "track_id": small.index,
        "genre": small[("track", "genre_top")].values,
    }).set_index("track_id")
    # drop rows without a genre label
    labels = labels.dropna(subset=["genre"])
    print(f"      -> labeled tracks: {len(labels)} across {labels['genre'].nunique()} genres")
    return labels

def subsample(labels):
    print(f"[4/5] Subsampling {TRACKS_PER_GENRE} tracks per genre ...")
    # group-wise sampling with a fixed seed -> same tracks every run
    sampled = labels.groupby("genre", group_keys=False).sample(
        n=TRACKS_PER_GENRE, random_state=SEED)
    print(f"      -> total sampled: {len(sampled)}")
    return sampled

def make_splits(labels):
    print("[5/5] Building stratified train/val/test splits (70/15/15) ...")
    # first carve off 15% for test, stratified by genre
    train_val, test = train_test_split(
        labels, test_size=0.15, stratify=labels["genre"], random_state=SEED)
    # then split remainder -> ~15% val, ~70% train
    train, val = train_test_split(
        train_val, test_size=0.1765, stratify=train_val["genre"], random_state=SEED)
    print(f"      -> train={len(train)}  val={len(val)}  test={len(test)}")
    return train, val, test